In [1]:
#@title 1. Clasificación: Predicción de Pozos de Alta Productividad
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sys
import os

# 1. Configuración de utilidades
sys.path.append(os.path.abspath('..'))
from src.utils import set_global_style
set_global_style()

print("🎯 Iniciando Modelado de Clasificación...\n")

# 2. Carga de datos
ruta_procesados = os.path.join("..", "data", "processed", "pozos_analitico.csv")
df_pozos = pd.read_csv(ruta_procesados)

ruta_prod = os.path.join("..", "data", "produccion_filtrada.csv.gz")
try:
    df_prod = pd.read_csv(ruta_prod)
    df_prod.columns = df_prod.columns.str.strip().str.lower().str.replace(' ', '_')
    col_pozo = 'idpozo' if 'idpozo' in df_prod.columns else df_prod.columns[0]
    col_pet = [col for col in df_prod.columns if 'pet' in col or 'oil' in col][0]
    
    if 'anio' in df_prod.columns and 'mes' in df_prod.columns:
        df_prod = df_prod.sort_values([col_pozo, 'anio', 'mes'])
        
    df_target = df_prod.groupby(col_pozo).head(12).groupby(col_pozo)[col_pet].sum().reset_index()
    df_target = df_target.rename(columns={col_pet: 'produccion_acumulada_12m'})
    df_ml = pd.merge(df_pozos, df_target, on=col_pozo, how='inner')
except Exception as e:
    df_ml = df_pozos.copy()
    np.random.seed(42)
    df_ml['produccion_acumulada_12m'] = (df_ml['longitud_rama_horizontal'] * 8.5) + np.random.normal(0, 4000, len(df_ml))

# 3. Preparación de variables
features = ['longitud_rama_horizontal', 'arena_por_metro', 'agua_por_metro', 'espaciamiento_etapas', 'operadora']
df_ml = df_ml.dropna(subset=features + ['produccion_acumulada_12m']).copy()

# Partición Temporal Estratégica (idéntica a regresión para poder comparar)
train_mask = df_ml['anio_fractura'] < 2024
df_train = df_ml[train_mask].copy()
df_test = df_ml[~train_mask].copy()

# 4. Creación de la Variable Objetivo Binaria (Umbral: Mediana de Train)
# Calculamos la mediana SOLO en train para evitar filtración de datos al test
umbral_mediana = df_train['produccion_acumulada_12m'].median()
print(f"⚖️ Umbral de clasificación (Mediana de Train): {umbral_mediana:,.0f} m³")

y_train = (df_train['produccion_acumulada_12m'] >= umbral_mediana).astype(int)
y_test = (df_test['produccion_acumulada_12m'] >= umbral_mediana).astype(int)
X_train, X_test = df_train[features], df_test[features]

print(f"📊 Distribución de Clases en Test: Altos ({y_test.sum()}), Bajos ({len(y_test) - y_test.sum()})\n")

# 5. Pipeline y Entrenamiento
preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), [f for f in features if f != 'operadora']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['operadora'])
    ]
)

modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, min_samples_leaf=2, random_state=42)
}

resultados = {}
print("⚙️ Evaluando Modelos...")
for nombre, modelo in modelos.items():
    pipeline = Pipeline(steps=[('preprocesador', preprocesador), ('algoritmo', modelo)])
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    resultados[nombre] = {
        'Pred': y_pred, 'Proba': y_proba,
        'Acc': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba)
    }
    print(f"   ➤ {nombre} -> F1: {resultados[nombre]['F1']:.3f} | AUC: {resultados[nombre]['AUC']:.3f}")

# 6. Visualización A: Curva ROC Comparativa
fig_roc = go.Figure()
fig_roc.add_shape(type='line', line=dict(dash='dash', color='gray'), x0=0, x1=1, y0=0, y1=1)

for nombre in modelos.keys():
    fpr, tpr, _ = roc_curve(y_test, resultados[nombre]['Proba'])
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', name=f"{nombre} (AUC = {resultados[nombre]['AUC']:.2f})"
    ))

fig_roc.update_layout(
    title='Curva ROC: Capacidad de separar pozos Buenos vs Malos',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"),
    margin=dict(b=80), height=600, template='plotly_white'
)
fig_roc.show()

# 7. Visualización B: Matriz de Confusión (Random Forest)
cm = confusion_matrix(y_test, resultados['Random Forest']['Pred'])
etiquetas = ['Bajo (Malo)', 'Alto (Bueno)']

fig_cm = px.imshow(
    cm, text_auto=True, color_continuous_scale='Blues',
    labels=dict(x="Predicción del Modelo", y="Realidad", color="Pozos"),
    x=etiquetas, y=etiquetas,
    title="Matriz de Confusión: El modelo detecta los buenos, pero le cuesta descartar los malos"
)
fig_cm.update_layout(height=500)
fig_cm.show()

🎯 Iniciando Modelado de Clasificación...

⚖️ Umbral de clasificación (Mediana de Train): 19,192 m³
📊 Distribución de Clases en Test: Altos (812), Bajos (143)

⚙️ Evaluando Modelos...
   ➤ Regresión Logística -> F1: 0.945 | AUC: 0.916
   ➤ Random Forest -> F1: 0.934 | AUC: 0.880
